In [1]:
import os
import pickle
import logging
import warnings
import pandas as pd
from prophet import Prophet
from joblib import Parallel, delayed
from tqdm import tqdm

# Suppress Prophet logs
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
warnings.filterwarnings('ignore')

OUTPUT_DIR = "pickled_models"

def refit_and_pickle(row_dict, df_data):
    """Reads winning parameters, builds a new model, trains it, and pickles it."""
    try:
        area = row_dict['Area']
        category = row_dict['Category']
        cps = row_dict['Best_CPS']
        cpr = row_dict['Best_CPR']

        # Isolate 100% historical data (1961-2019) for this specific region/season
        sub_df = df_data[(df_data['Area'] == area) & (df_data['Months'] == category)]
        
        prophet_df = pd.DataFrame({
            'ds': pd.to_datetime(sub_df['Year'].astype(str) + '-01-01'),
            'y': sub_df['Temperature_Change'].values
        })

        # 1. Build the NEW model using the optimal parameters from the registry
        final_model = Prophet(
            yearly_seasonality=False,
            weekly_seasonality=False,
            daily_seasonality=False,
            changepoint_prior_scale=cps,
            changepoint_range=cpr
        )
        
        # 2. Train the model on 100% of the data
        final_model.fit(prophet_df)

        # 3. Format the filename and Pickle the model object
        safe_area = str(area).replace(' ', '_').replace('/', '_')
        safe_cat = str(category).replace(' ', '_')
        pkl_filename = f"Prophet_{safe_area}_{safe_cat}.pkl"
        pkl_filepath = os.path.join(OUTPUT_DIR, pkl_filename)

        with open(pkl_filepath, 'wb') as f:
            pickle.dump(final_model, f)

        return pkl_filepath
    except Exception:
        return None

if __name__ == "__main__":
    # Read from the best registry file
    print("Loading optimal parameters from registry...")
    best_params_df = pd.read_csv("best_params_registry.csv")
    
    print("Loading historical dataset...")
    df = pd.read_csv("Cleaned_Environment_Temperature_Data.csv")
    
    # Standardize the season text to match the registry
    season_mapping = {
        'Dec\x96Jan\x96Feb': 'Winter', 'Mar\x96Apr\x96May': 'Spring',
        'Jun\x96Jul\x96Aug': 'Summer', 'Sep\x96Oct\x96Nov': 'Autumn'
    }
    df['Months'] = df['Months'].replace(season_mapping)

    # Create the output directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    print(f"Building and pickling {len(best_params_df)} new models...")

    # Convert dataframe to a list of dictionaries for the parallel loop
    rows = best_params_df.to_dict('records')
    
    # Execute the pickling across CPU cores
    results = Parallel(n_jobs=-1)(
        delayed(refit_and_pickle)(row, df)
        for row in tqdm(rows, desc="Pickling Models")
    )

    # Save a final summary file that includes the filepaths of your newly pickled models
    best_params_df['Model_Filepath'] = results
    best_params_df.to_csv("model_registry_summary.csv", index=False)

    print(f"\n=== PICKLING COMPLETE ===")
    print(f"All models successfully built and saved to directory: '{OUTPUT_DIR}/'")

d:\ml\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading optimal parameters from registry...
Loading historical dataset...
Building and pickling 3162 new models...


Pickling Models: 100%|██████████| 3162/3162 [03:06<00:00, 16.96it/s]



=== PICKLING COMPLETE ===
All models successfully built and saved to directory: 'pickled_models/'
